# Lesson 16 | Why can moving data be harder than adding?

The earlier **event-driven processing** model seems to do mostly additions and comparisons. At larger scale a common problem becomes:

> **The compute unit is waiting because data has not arrived yet.**

Primary new concept: **memory hierarchy and data movement can limit throughput, the sustained amount of work completed per unit time.**

## 1. Concept ledger

**Already known:** neuron state, synapse records, host and programmable logic (PL), event-driven processing.

**New today:** **memory hierarchy**, **latency**, **throughput**, and **bandwidth**.

**Preview only:** external memory and a standard on-chip communication protocol are separated into the next two lessons.

## 2. Why does this matter to FlyBrain?

After a source spike the system reads a source index and synapse records. Adding a weight can be cheap while delivering many records can be expensive.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 115" role="img" aria-label="l16-flow" style="width:100%;height:auto"><defs><marker id="l16-flow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs><rect x="10" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="100" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">synapse storage</text><path d="M190,57 h25" fill="none" stroke="#506070" marker-end="url(#l16-flow)"/><rect x="220" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="310" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">transfer bytes</text><path d="M400,57 h25" fill="none" stroke="#506070" marker-end="url(#l16-flow)"/><rect x="430" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="520" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">compute</text><path d="M610,57 h25" fill="none" stroke="#506070" marker-end="url(#l16-flow)"/><rect x="640" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="730" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">target state</text></svg>

If the arrow cannot deliver data fast enough, arithmetic units may sit idle.

## 3. Do not collapse the three metrics

- latency: how long one operation waits;
- throughput: sustained work completed per unit time;
- bandwidth: sustained data moved per unit time.

They interact but are not interchangeable.

## 4. A minimal cost model

For now, estimate two costs **separately** rather than pretending they are a complete model of total runtime:

- compute time = items × compute time/item;
- transfer time = startup latency + bytes ÷ bandwidth;
- the larger term is treated as the current **dominant cost / bottleneck candidate**.

If a real process serializes compute and transfer completely, elapsed time would be their sum. If they overlap, steady-state throughput needs a different model. This lesson only compares the two component costs; pipeline overlap is deferred.

## 5. Run: arithmetic or movement?

Predict which time is larger.

### Before running: what memory hierarchy contains

An FPGA can keep small state in registers, larger addressed state in on-chip **Block RAM (BRAM)**, and large connection tables in external DDR. External storage requires controllers and interconnect. Each has capacity, port-count, latency, and bandwidth limits.

One thousand 16-byte records total 16000 bytes. At 4 bytes/ns (4 GB/s), transfer takes 4000 ns plus 100 ns startup, or 4100 ns. Calculation takes 1000 ns. These are separate costs; a fully serial execution takes 5100 ns.


In [ ]:
num_items = 1000
bytes_per_item = 16
compute_ns_per_item = 1.0
bandwidth_bytes_per_ns = 4.0
startup_latency_ns = 100.0

compute_ns = num_items * compute_ns_per_item
transfer_ns = startup_latency_ns + (num_items * bytes_per_item) / bandwidth_bytes_per_ns

print("compute time:", compute_ns, "ns")
print("data-movement time:", transfer_ns, "ns")
print("dominant cost:", "memory/data movement" if transfer_ns > compute_ns else "compute")


## 6. Observe

Moving 16,000 bytes at 4 bytes/ns takes 4,000 ns plus 100 ns startup; arithmetic takes 1,000 ns. This example mostly waits for data.

## 7. Try It

Raise bandwidth from 4.0 to 32.0 and predict whether the bottleneck changes. Then increase compute time/item.

## 8. Exercise

[Lesson 16 exercise: identify the compute or data-movement bottleneck](../../exercises/en/16_data_movement_cost.ipynb)

## 9. AI Task

Ask an AI to reason from event count, bytes/event, bandwidth, and compute time. Check units and make sure latency and bandwidth are not treated as the same quantity.

## 10. Human Check

Explain why memory hierarchy matters to an event-driven spiking neural network (SNN); distinguish latency, throughput, and bandwidth; and explain why a fast adder does not guarantee high system throughput.

## 11. Engineering Handoff

Maps to `RMD-013A`: build memory-hierarchy and bandwidth intuition before DDR/AXI and describe movement cost with measurable quantities.

## 12. Project Trace

- Lesson: `LSN-016`
- Mapping: `RMD-013A`
- Metrics: latency / throughput / bandwidth
- Boundary: no DDR protocol details yet

## 13. Exit Ticket

Given compute and transfer parameters, you can identify the likely bottleneck and state the assumptions behind that conclusion.